<a href="https://colab.research.google.com/github/ArmandoBarrios/unidad-3-/blob/main/practica_3_unidad_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Machine learning
#Unidad 3
# Practica 3: KNN en Prestamos Lending Club
#Facilitador: Dr. José Gabriel Rodríguez Rivas
#Alumno:Armando Barrios

#Clasificador K-Nearest Neighbors (KNN)
El algoritmo K-Nearest Neighbors se puede utilizar para la clasificación y regresión. Los clasificadores k-nn son algoritmos de aprendizaje supervisado basados en instancias o basados en memoria. Los métodos de aprendizaje basados en instancias, funcionan memorizando los ejemplos etiquetados que ven en el conjunto de entrenamiento, y luego, usan esos ejemplos memorizados para clasificar nuevos objetos más adelante.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

#Carga el Dataset
from google.colab import drive
drive.mount('/content/drive')
#ID del archivo
#https:https:https://drive.google.com/file/d/1V5ANwclpm0o4aHH_HvFeD0P0db0rqZOO/view?usp=sharing
file_id = '1V5ANwclpm0o4aHH_HvFeD0P0db0rqZOO'
url =  f"https://drive.google.com/uc?id={file_id}"
# Importamos librerias necesarias para knn
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
prestamos_df = pd.read_csv(url)
prestamos_df.head()

,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,...,application_type,acc_now_delinq,chargeoff_within_12_mths,delinq_amnt,pub_rec_bankruptcies,tax_liens,hardship_flag,disbursement_method,debt_settlement_flag,debt_settlement_flag_date
0,2400,2400,2400.0,36 months,15.96,84.33,C,C5,NaN,10+ years,...,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N,NaN
1,10000,10000,10000.0,36 months,13.49,339.31,C,C1,AIR RESOURCES BOARD,10+ years,...,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N,NaN
2,3000,3000,3000.0,36 months,18.64,109.43,E,E1,MKC Accounting,9 years,...,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N,NaN
3,5600,5600,5600.0,60 months,21.28,152.39,F,F2,NaN,4 years,...,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N,NaN
4,5375,5375,5350.0,60 months,12.69,121.45,B,B5,Starbucks,< 1 year,...,Individual,0.0,0.0,0.0,0.0,0.0,N,Cash,N,NaN


#Selección de Variables predictoras
Para la seleccion de caracteristicas que se utilizarán en el modelo, la mejor opción es comprender los datos, y comprender las reglas del negocio Selecionamos las siguientes variables predictoras (columnas)

funded_amnt ( monto financiado) La cantidad total comprometida con ese préstamo en ese momento.

loan_term_year ( Plazo del credito ) int_rate ( taza de interes ) grade_code ( codigo de la calificacion del crédito ) purpose_code ( codigo del proposito) addr_state_code ( codigo de la ciudad) home_ownership_code ( codigo situacion casa) annual_inc ( ingresos anuales) dti ( Una relación calculada utilizando los pagos de deuda mensuales totales del prestatario ) revol_util ( cantidad de crédito que el prestatario está utilizando en relación con todo el crédito ) pub_rec_bankruptcies ( Número de quiebras de registros públicos ) Variable dependiente (variable a predecir) la etiqueta repaid será la elegida para el modelo de clasificación. Otros aspectos a considerar Solo las características que están disponibles antes de que se inicie el préstamo pueden usarse en la clasificación. Las características como recoveries (recuperaciones o cargo posterior a la recuperación bruta), total_rec_prncp (Principal recibido a la fecha), que solo están disponibles después de que se cierra el préstamo, no deben incluirse en las funciones de entrenamiento. Si la clasificación logra una tasa de precisión cercana al 100%, es probable que incluya características que solo estarán disponibles después de que se cierre el préstamo.

In [ ]:
# Create coded versions of categorical columns
prestamos_df['grade_code'] = prestamos_df['grade'].astype('category').cat.codes
prestamos_df['purpose_code'] = prestamos_df['purpose'].astype('category').cat.codes
prestamos_df['addr_state_code'] = prestamos_df['addr_state'].astype('category').cat.codes
prestamos_df['home_ownership_code'] = prestamos_df['home_ownership'].astype('category').cat.codes

# Create the 'repaid' target variable from 'loan_status'
# Assuming 'Fully Paid' means repaid (1) and anything else means not repaid (0)
prestamos_df['repaid'] = prestamos_df['loan_status'].apply(lambda status: 1 if status == 'Fully Paid' else 0)

X = prestamos_df[['funded_amnt', "int_rate", "grade_code", 'purpose_code', 'addr_state_code', 'home_ownership_code', 'annual_inc', 'dti', 'revol_util', 'pub_rec_bankruptcies']] # Variable objetivo o variable a predecir
y = prestamos_df["repaid"]

#Dividir dataset en conjunto de entrenamiento y prueba

In [ ]:
# Dividimos el dataFrame df en df_train y df_test.
# 60% para dataset de entrenamient0, y 40% para dataset de prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42)

# verificamos la cantidad de registros asignados al dataframe de entrenamiento
X_train.shape, X_test.shape, y_train.shape, y_test.shape
((11944, 10), (7964, 10), (11944,), (7964,))

((11944, 10), (7964, 10), (11944,), (7964,))

#Modelo de Clasificacion KNN con 1 vecino


In [ ]:
# Creamoos el clasificador de regresion logistica
knn1 = KNeighborsClassifier(n_neighbors = 1)

# Imputar valores nulos en X_train y X_test antes de entrenar
# Usamos SimpleImputer para reemplazar NaN con la mediana de cada columna.
from sklearn.impute import SimpleImputer

# Create an imputer object with a median strategy
imputer = SimpleImputer(strategy='median')

# Impute X_train and X_test, ensuring they remain pandas DataFrames
# Fit the imputer on X_train and transform X_train
X_train_imputed_array = imputer.fit_transform(X_train)
X_train = pd.DataFrame(X_train_imputed_array, columns=X_train.columns, index=X_train.index)

# Transform X_test using the imputer fitted on X_train
X_test_imputed_array = imputer.transform(X_test)
X_test = pd.DataFrame(X_test_imputed_array, columns=X_test.columns, index=X_test.index)

# Entrenamos el clasificador
knn1.fit(X_train, y_train)
# Precisión del modelo en la fase de entrenamiento
print("Precision del clasificador en fase de entrenamiento", knn1.score(X_train, y_train) )

Precision del clasificador en fase de entrenamiento 1.0


In [ ]:
# Realizar una prediccion con los datos de prueba
y_pred = knn1.predict(X_test)
# Crear un informe de texto que muestre las principales métricas de clasificación.
print("\nReporte de métricas del clasificador con 1 vecino: \n", classification_report(y_test, y_pred, target_names=["No Pagado", "Pagado"]))
print(f'\nMatriz Confusion con 1 vecino:\n', confusion_matrix(y_test, y_pred ))


Reporte de métricas del clasificador con 1 vecino: 
               precision    recall  f1-score   support

   No Pagado       0.17      0.15      0.16      1220
      Pagado       0.85      0.86      0.86      6744

    accuracy                           0.76      7964
   macro avg       0.51      0.51      0.51      7964
weighted avg       0.74      0.76      0.75      7964


Matriz Confusion con 1 vecino:
 [[ 185 1035]
 [ 915 5829]]


#Interpretacion
184 → verdaderos “No Pagados” correctamente detectados (TP clase minoritaria) 1036 → falsos negativos (prestamos impagos clasificados como pagados) 915 → falsos positivos (pagados clasificados como impagos) Detecta algunos impagos, pero aún confunde muchos. El balance entre precisión y sensibilidad es débil, pero hay algo de detección útil.

#Modelo de Clasificacion KNN con 5 vecinos

In [ ]:
# Creamoos el clasificador de regresion logistica
knn5 = KNeighborsClassifier(n_neighbors = 5)
# Entrenamos el clasificador
knn5.fit(X_train, y_train)
# Precisión del modelo en la fase de entrenamiento
print("Precision del clasificador en fase de entrenamiento", knn5.score(X_train, y_train) )

Precision del clasificador en fase de entrenamiento 0.8614367046215673


In [ ]:
# Realizar una prediccion con los datos de prueba
y_pred = knn5.predict(X_test)
# Crear un informe de texto que muestre las principales métricas de clasificación.
print("\nReporte de métricas del clasificador con 5 vecinos: \n", classification_report(y_test, y_pred, target_names=["No Pagado", "Pagado"]))
print(f'\nMatriz de Confusion con 5 vecinos:\n', confusion_matrix(y_test, y_pred ))


Reporte de métricas del clasificador con 5 vecinos: 
               precision    recall  f1-score   support

   No Pagado       0.21      0.04      0.06      1220
      Pagado       0.85      0.98      0.91      6744

    accuracy                           0.83      7964
   macro avg       0.53      0.51      0.48      7964
weighted avg       0.75      0.83      0.78      7964


Matriz de Confusion con 5 vecinos:
 [[  43 1177]
 [ 166 6578]]


#Interpretacion
Solo detecta 43 impagos reales de 1220. El modelo prácticamente ha olvidado la clase minoritaria.

#Modelo de Clasificacion KNN con 10 vecinos

In [ ]:
# Creamoos el clasificador de regresion logistica
knn10 = KNeighborsClassifier(n_neighbors = 10)
# Entrenamos el clasificador
knn10.fit(X_train, y_train)
# Precisión del modelo en la fase de entrenamiento
print("Precision del clasificador en fase de entrenamiento", knn10.score(X_train, y_train) )

Precision del clasificador en fase de entrenamiento 0.8544876088412592


In [ ]:
# Realizar una prediccion con los datos de prueba
y_pred = knn10.predict(X_test)
# Crear un informe de texto que muestre las principales métricas de clasificacion
print("\nReporte de métricas del clasificador con 10 vecinos: \n", classification_report(y_test, y_pred, target_names=["No Pagado", "Pagado"]))
print(f'Matriz de Confusion con 10 vecinos:\n', confusion_matrix(y_test, y_pred ))


Reporte de métricas del clasificador con 10 vecinos: 
               precision    recall  f1-score   support

   No Pagado       0.20      0.02      0.03      1220
      Pagado       0.85      0.99      0.91      6744

    accuracy                           0.84      7964
   macro avg       0.52      0.50      0.47      7964
weighted avg       0.75      0.84      0.78      7964

Matriz de Confusion con 10 vecinos:
 [[  19 1201]
 [  77 6667]]


Interpretación Este modelo es peor: solo detecta 19 impagos. La red de vecinos se ha “suavizado” tanto que todas las predicciones se van a la clase mayoritaria (Pagado). Recomendaciones   KNN depende de distancias. Si las variables no están escaladas, las de mayor rango (por ejemplo, annual_inc) dominan las distancias. KNN es muy sensible al desbalance: cuando busca los k vecinos más cercanos, la mayoría pertenecen a la clase mayoritaria (“Pagado”), por lo tanto el voto final también. Balancear las clases. Oversampling (replicar casos de “No Pagado”)
# Importancia de las variables

In [ ]:
from sklearn.inspection import permutation_importance

result = permutation_importance(knn10, X_test, y_test, n_repeats=30, random_state=42)
importancia = pd.DataFrame({
    "Variable": X.columns,
    "Importancia Media": result.importances_mean,
    "Desviación": result.importances_std
}).sort_values("Importancia Media", ascending=False)
print(importancia)

               Variable  Importancia Media  Desviación
0           funded_amnt           0.001641    0.001229
6            annual_inc           0.001431    0.001172
8            revol_util           0.000586    0.000429
7                   dti           0.000146    0.000218
2            grade_code           0.000054    0.000077
1              int_rate           0.000033    0.000121
4       addr_state_code           0.000008    0.000107
5   home_ownership_code           0.000000    0.000000
9  pub_rec_bankruptcies           0.000000    0.000000
3          purpose_code          -0.000067    0.000155


In [ ]:
# Visualizar importancias con Plotly
import plotly.express as px
fig4 = px.bar(importancia, x="Variable", y="Importancia Media", error_y="Desviación", title="Importancia de características (KNN - Permutation Importance)", text_auto=".3f", color="Variable")
fig4.update_layout(width=800, height=600)
fig4.show()

#Normalización de los datos con StandardScaler

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer # Import SimpleImputer

X = prestamos_df[['funded_amnt', "int_rate", "grade_code", 'purpose_code', 'addr_state_code', 'home_ownership_code', 'annual_inc', 'dti', 'revol_util', 'pub_rec_bankruptcies']]
y = prestamos_df["repaid"]

# Dividir Datos, stratify es para que mantenga la misma proporción de clases en ambos conjuntos
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)

# Imputar valores nulos en X_train y X_test antes de escalar
imputer = SimpleImputer(strategy='median')
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

# Normalizar variables (KNN es sensible a la escala) *after* imputation
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

#Modelo de Clasificación con datos normalizados y 3 vecinos

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# Entrenar KNN
knn_scaled = KNeighborsClassifier(n_neighbors=3)

# Re-imputar y escalar los datos justo antes de entrenar
# (Esto se incluye aquí para hacer la celda autocontenida y robusta a problemas de ejecución previa)
imputer_local = SimpleImputer(strategy='median')
X_train_imputed_local = imputer_local.fit_transform(X_train)
X_test_imputed_local = imputer_local.transform(X_test)

scaler_local = StandardScaler()
X_train_scaled_local = scaler_local.fit_transform(X_train_imputed_local)
X_test_scaled_local = scaler_local.transform(X_test_imputed_local)

knn_scaled.fit(X_train_scaled_local, y_train)
y_pred_scaled = knn_scaled.predict(X_test_scaled_local)
print("\nReporte de clasificación datos normalizados y 3 vecinos:")
print(classification_report(y_test, y_pred_scaled, target_names=["No Pagado", "Pagado"]))
print ("\nMatriz de confusión con datos normalizados y 3 vecinos:\n", confusion_matrix(y_test, y_pred_scaled), "\n")


Reporte de clasificación datos normalizados y 3 vecinos:
              precision    recall  f1-score   support

   No Pagado       0.22      0.10      0.14      1177
      Pagado       0.86      0.94      0.89      6787

    accuracy                           0.81      7964
   macro avg       0.54      0.52      0.52      7964
weighted avg       0.76      0.81      0.78      7964


Matriz de confusión con datos normalizados y 3 vecinos:
 [[ 121 1056]
 [ 441 6346]] 



Interpretación 121 verdaderos impagos detectados de 1177 reales → recall = 10% 1056 impagos mal clasificados como pagados (falsos negativos) 441 pagados mal clasificados como impagos (falsos positivos) El modelo identif ica correctamente casi todos los préstamos buenos (Pagado), pero falla en detectar los malos (No Pagado), lo que en contexto financiero es el peor tipo de error, ya que un préstamo riesgoso se aprueba por error.
#Modelo de Clasificación con datos normalizados y 5 vecinos

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# Entrenar KNN
knn_scaled = KNeighborsClassifier(n_neighbors=5)

# Re-imputar y escalar los datos justo antes de entrenar
# (Esto se incluye aquí para hacer la celda autocontenida y robusta a problemas de ejecución previa)
imputer_local = SimpleImputer(strategy='median')
X_train_imputed_local = imputer_local.fit_transform(X_train)
X_test_imputed_local = imputer_local.transform(X_test)

scaler_local = StandardScaler()
X_train_scaled_local = scaler_local.fit_transform(X_train_imputed_local)
X_test_scaled_local = scaler_local.transform(X_test_imputed_local)

knn_scaled.fit(X_train_scaled_local, y_train)
y_pred_scaled = knn_scaled.predict(X_test_scaled_local)
print("\nReporte de clasificación datos normalizados y 5 vecinos:")
print(classification_report(y_test, y_pred_scaled, target_names=["No Pagado", "Pagado"]))
print ("\nMatriz de confusión con datos normalizados y 5 vecinos:\n", confusion_matrix(y_test, y_pred_scaled), "\n")


Reporte de clasificación datos normalizados y 5 vecinos:
              precision    recall  f1-score   support

   No Pagado       0.24      0.07      0.11      1177
      Pagado       0.86      0.96      0.91      6787

    accuracy                           0.83      7964
   macro avg       0.55      0.52      0.51      7964
weighted avg       0.76      0.83      0.79      7964


Matriz de confusión con datos normalizados y 5 vecinos:
 [[  80 1097]
 [ 257 6530]] 



#interpretación
80 verdaderos impagos detectados de 1177 reales → recall = 7% 1097 impagos mal clasificados como pagados (falsos negativos) 257 pagados mal clasificados como impagos (falsos positivos)
#Modelo de Clasificación datos normalizados y 10 vecinos

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# Entrenar KNN
knn_scaled = KNeighborsClassifier(n_neighbors=10)

# Re-imputar y escalar los datos justo antes de entrenar
# (Esto se incluye aquí para hacer la celda autocontenida y robusta a problemas de ejecución previa)
imputer_local = SimpleImputer(strategy='median')
X_train_imputed_local = imputer_local.fit_transform(X_train)
X_test_imputed_local = imputer_local.transform(X_test)

scaler_local = StandardScaler()
X_train_scaled_local = scaler_local.fit_transform(X_train_imputed_local)
X_test_scaled_local = scaler_local.transform(X_test_imputed_local)

knn_scaled.fit(X_train_scaled_local, y_train)
y_pred_scaled = knn_scaled.predict(X_test_scaled_local)
print("\nReporte de clasificación datos normalizados y 10 vecinos:")
print(classification_report(y_test, y_pred_scaled, target_names=["No Pagado", "Pagado"]))
print ("\nMatriz de confusión con datos normalizados y 10 vecinos:\n", confusion_matrix(y_test, y_pred_scaled), "\n")


Reporte de clasificación datos normalizados y 10 vecinos:
              precision    recall  f1-score   support

   No Pagado       0.25      0.05      0.08      1177
      Pagado       0.86      0.97      0.91      6787

    accuracy                           0.84      7964
   macro avg       0.55      0.51      0.50      7964
weighted avg       0.77      0.84      0.79      7964


Matriz de confusión con datos normalizados y 10 vecinos:
 [[  57 1120]
 [ 172 6615]] 



Interpretacion general de resultados
terpretacion general de resultados Sin normalización Vecinos (k) Exactitud Recall No Pagado Recall Pagado Observaciones 1 5 10 0.76 0.83 0.84 Con normalizacion 0.15 0.04 0.02 0.86 0.98 0.99 Vecinos (k) Exactitud Recall No Pagado Recall Pagado Detecta mejor los “Pagados”, pero pierde muchos “No Pagados”. Mucho mejor exactitud, pero casi no detecta “No Pagados” → desequilibrio fuerte. Muy alto recall en “Pagados”, pero ignora los “No Pagados” casi por completo. Observaciones 3 5 10 0.81 0.83 0.84 0.10 0.07 0.05 0.94 0.96 0.97 Mejora leve el equilibrio. Buen desempeño general, aunque sigue priorizando “Pagado”. Similar al anterior, pero con aún menos sensibilidad a los “No Pagados”. Qué significan las métricas en este contexto financiero Recall (No Pagado) = sensibilidad para detectar préstamos de alto riesgo. → Muy bajo → el banco subestima el riesgo, puede aprobar clientes morosos. Precision (No Pagado) = de los préstamos que el modelo clasifica como “No Pagado”, ¿cuántos realmente lo son? → Aumenta un poco al normalizar (0.22–0.25), pero sigue bajo. Accuracy alto (~0.83) es engañoso: refleja solo la clase mayoritaria (“Pagado”), no la capacidad de detectar morosos. Mejor modelo KNN actual: con normalización y k=3. Aunque el rendimiento global no es excelente, es el único que mantiene cierto equilibrio

# Abordando el desbalance de clases con SMOTE

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

# Asegurarse de que X y y estén actualizados con las variables seleccionadas
X = prestamos_df[['funded_amnt', "int_rate", "grade_code", 'purpose_code', 'addr_state_code', 'home_ownership_code', 'annual_inc', 'dti', 'revol_util', 'pub_rec_bankruptcies']]
y = prestamos_df["repaid"]

# Dividir Datos (con stratify para mantener la proporción antes de SMOTE)
X_train_smote, X_test_smote, y_train_smote, y_test_smote = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)

# Imputar valores nulos antes de SMOTE y escalado
imputer_smote = SimpleImputer(strategy='median')
X_train_imputed_smote = imputer_smote.fit_transform(X_train_smote)
X_test_imputed_smote = imputer_smote.transform(X_test_smote)

# Escalar los datos imputados
scaler_smote = StandardScaler()
X_train_scaled_smote = scaler_smote.fit_transform(X_train_imputed_smote)
X_test_scaled_smote = scaler_smote.transform(X_test_imputed_smote)

# Aplicar SMOTE solo al conjunto de entrenamiento y a los datos escalados
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled_smote, y_train_smote)

print(f"Dimensiones de X_train_resampled: {X_train_resampled.shape}")
print(f"Conteo de clases en y_train_resampled:\n{pd.Series(y_train_resampled).value_counts()}")


Dimensiones de X_train_resampled: (20356, 10)
Conteo de clases en y_train_resampled:
repaid
0    10178
1    10178
Name: count, dtype: int64


# Modelo de Clasificación KNN con datos normalizados y SMOTE (k=3)

In [ ]:
# Entrenar KNN con los datos balanceados por SMOTE
knn_smote = KNeighborsClassifier(n_neighbors=3) # Usamos k=3 como el modelo 'mejor' anterior
knn_smote.fit(X_train_resampled, y_train_resampled)

# Realizar predicciones en el conjunto de prueba (que no fue sobremuestreado)
y_pred_smote = knn_smote.predict(X_test_scaled_smote)

# Evaluar el modelo
print("\nReporte de clasificación con SMOTE y 3 vecinos:")
print(classification_report(y_test_smote, y_pred_smote, target_names=["No Pagado", "Pagado"]))
print ("\nMatriz de confusión con SMOTE y 3 vecinos:\n", confusion_matrix(y_test_smote, y_pred_smote), "\n")


Reporte de clasificación con SMOTE y 3 vecinos:
              precision    recall  f1-score   support

   No Pagado       0.19      0.42      0.26      1177
      Pagado       0.87      0.69      0.77      6787

    accuracy                           0.65      7964
   macro avg       0.53      0.55      0.51      7964
weighted avg       0.77      0.65      0.69      7964


Matriz de confusión con SMOTE y 3 vecinos:
 [[ 494  683]
 [2124 4663]] 

